In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

from scipy.io import mmread, mmwrite
from scipy.sparse import csr_matrix, vstack
from scipy.stats import ttest_ind, mannwhitneyu, zscore

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/atac/peaks/clusters/justin_chromhmm_cluster_donor.h5ad')

In [ ]:
adata

In [ ]:
### raw counts
adata.X

In [ ]:
tmp_mtx = pd.DataFrame(adata.X)
cpm = tmp_mtx.div(tmp_mtx.sum(axis=1), axis=0) * 1e6
adata.X = cpm.values

In [ ]:
adata.X.sum(axis=0), \
adata.X.sum(axis=1)

In [ ]:
adata.X.min()

In [ ]:
adata.X = (adata.X - adata.X.mean(axis=0, keepdims=True)) / adata.X.std(axis=0, keepdims=True)

In [ ]:
adata.X.mean(axis=0), \
adata.X.mean(axis=1)

In [ ]:
# norm_mtx

In [ ]:
adata.var.index = [f'{x}_{y}' for x,y in zip(adata.var['seqnames'], adata.var['start'])]

In [ ]:
s = 'Start'
fc = pd.read_csv(f'{projdir}/csv/atac/peaks/chromhmm/coarse_donor_peaks.fc.{s}.csv', sep='\t')
pvals = pd.read_csv(f'{projdir}/csv/atac/peaks/chromhmm/coarse_donor_peaks.pvals.{s}.csv', sep='\t')
fc.shape, pvals.shape

In [ ]:
fc.head()

In [ ]:
pvals.head()

In [ ]:
cutoff = 0.05
for i in range(4):
    print((pvals.iloc[:,i] < cutoff).sum())

In [ ]:
donors = ['C29', 'C37', 'C38', 'C39']

In [ ]:
i = 0
s = donors[i]

pval_cutoff = 0.01
fc_cutoff = 1

mask = (pvals.iloc[:,i] < pval_cutoff) & (fc.iloc[:,i] > fc_cutoff)
tmp_peaks = pvals[mask].index

mask = adata.var.index.isin(tmp_peaks)
tmp_adata = adata[:, mask]

tmp_mtx = pd.DataFrame(tmp_adata.X, index=tmp_adata.obs['cluster_donor'])
norm_mtx = tmp_mtx.apply(zscore)
tmp_adata.X = norm_mtx.values
tmp_adata

In [ ]:
peak_dist = pd.DataFrame(tmp_adata.X).T
peak_dist.columns = tmp_adata.obs['cluster_donor']
to_plot = peak_dist.melt(value_vars=peak_dist.columns)
to_plot['cluster'] = [x.split('_')[0] for x in to_plot['cluster_donor']]
to_plot['donor'] = [x.split('_')[1] for x in to_plot['cluster_donor']]
to_plot.head()

In [ ]:
donor_peak_results = []

for i in range(4):
    d = donors[i]
    mask = (pvals.iloc[:,i] < pval_cutoff) & (fc.iloc[:,i] > fc_cutoff)
    tmp_peaks = pvals[mask].index
    print(tmp_peaks.shape)
    
    mask = adata.var.index.isin(tmp_peaks)
    tmp_adata = adata[:, mask]
    
    tmp_mtx = pd.DataFrame(tmp_adata.X, index=tmp_adata.obs['cluster_donor'])
    norm_mtx = tmp_mtx.apply(zscore)
    tmp_adata.X = norm_mtx.values
    
    peak_dist = pd.DataFrame(tmp_adata.X).T
    peak_dist.columns = tmp_adata.obs['cluster_donor']
    
    peak_dist['plot_donor'] = d
    donor_peak_results.append(peak_dist)

In [ ]:
### I think this is all I need

tmp_results = []

for i, s in enumerate(donors):
    d = donors[i]
    peak_dist = donor_peak_results[i].iloc[:,:-1]
    
    to_plot = peak_dist.melt(value_vars=peak_dist.columns)
    to_plot['cluster'] = [x.split('_')[0] for x in to_plot['cluster_donor']]
    to_plot['donor'] = [x.split('_')[1] for x in to_plot['cluster_donor']]

    fig, axes = plt.subplots(1, figsize=(6, 4))

    x_lab = 'cluster'
    y_lab = 'value'
    hue = 'donor'

    ax = sns.boxplot(data=to_plot, x=x_lab, y=y_lab, hue=hue, order=tmp_order, showfliers=False)
    ax.set_xlabel('', fontsize=12)
    ax.set_ylabel('', fontsize=12)
    ax.set_title('', fontsize=20)

    ax.tick_params(axis='x', labelsize=14, labelrotation=45)
    sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
    
    to_plot['plot_donor'] = 'rest'
    mask = to_plot['donor'] == d
    to_plot.loc[mask, 'plot_donor'] = 'target'
    tmp_results.append(to_plot)

In [ ]:
target_plot = pd.concat(tmp_results)
fig, ax = plt.subplots(1, figsize=(8, 4))

x_lab = 'cluster'
y_lab = 'value'
hue = 'plot_donor'

sns.boxplot(data=target_plot, x=x_lab, y=y_lab, hue=hue, showfliers=False, ax=ax)
ax.set_xlabel('', fontsize=12)
ax.set_ylabel('', fontsize=12)
ax.set_title(f'DEGs over trajectory', fontsize=16)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
merged = pd.concat(tmp_results)
merged.head()

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure2'
target_plot.to_csv(f'{outdir}/atac_start_donor_peaks_over_traj.csv', sep='\t')

### pvalues

In [ ]:
all_pvals = []
pval_cutoff = 0.01
fc_cutoff = 1

for i in range(4):
    donor = donors[i]
    tmp_pvals = []

    mask = (pvals.iloc[:,i] < pval_cutoff) & (fc.iloc[:,i] > fc_cutoff)
    tmp_peaks = pvals[mask].index

    mask = adata.var.index.isin(tmp_peaks)
    tmp_adata = adata[:, mask]
    
    tmp_mtx = pd.DataFrame(tmp_adata.X, index=tmp_adata.obs['cluster_donor'])
    norm_mtx = tmp_mtx.apply(zscore)
    tmp_adata.X = norm_mtx.values

    peak_dist = pd.DataFrame(tmp_adata.X).T
    peak_dist.columns = tmp_adata.obs['cluster_donor']

    tmp_df = peak_dist.T

    for s in tmp_order:
        mask = [x.split('_')[0] == s for x in tmp_df.index]
        clust_df = tmp_df.loc[mask,:]
        mask = np.array([x.split('_')[1] == donor for x in clust_df.index])
        donor_vals = clust_df.loc[mask].values.reshape(-1,1)
        rest_vals = clust_df.loc[~mask].mean(axis=0).values.reshape(-1,1)
        tmp_pvals.append(ttest_ind(a=donor_vals, b=rest_vals).pvalue[0])
        
    all_pvals.append(tmp_pvals)
pval_df  = pd.DataFrame(all_pvals, index=donors, columns=tmp_order)
pval_df.shape

In [ ]:
pval_df

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure2'
pval_df.to_csv(f'{outdir}/atac_start_donor_peaks_over_traj_pvals.csv', sep='\t')

### can think about doing a target donor vs. rest if needed

In [ ]:
# ### 

In [ ]:
# ### try target donor vs. 
    

In [ ]:
!date